# Notebook 2: Building a Knowledge Graph from Plain Text

In this notebook, you'll learn how to:
1. Define a **schema** (what types of entities and relationships to extract)
2. Use `SimpleKGPipeline` to extract knowledge from plain text
3. Explore the resulting graph in Neo4j

## How It Works

The pipeline follows these steps:

```
Text → Split into chunks → LLM extracts entities & relationships → Write to Neo4j → Resolve duplicates
```

The LLM reads each chunk of text and identifies entities (nodes) and their relationships, then writes them to the graph database.

## Step 1: Setup (same as Notebook 1)

In [1]:
import os
import asyncio
import neo4j
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")

NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")

driver = neo4j.GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD),
)
driver.verify_connectivity()
print("Connected to Neo4j!")

Connected to Neo4j!


## Step 2: Prepare the LLM and Embedder

We need two AI components:
- **LLM**: Reads text and extracts entities/relationships
- **Embedder** (text-embedding-3-large): Creates vector representations of text chunks for later retrieval

The `model_params` follow the [KG Builder documentation](https://neo4j.com/docs/neo4j-graphrag-python/current/user_guide_kg_builder.html):
- `max_completion_tokens`: limits the response length
- `response_format: json_object`: ensures the LLM returns valid JSON (recommended for the entity extractor)

In [ ]:
from neo4j_graphrag.llm import OpenAILLM
from neo4j_graphrag.embeddings import OpenAIEmbeddings

llm = OpenAILLM(
    model_name="gpt-4.1-2025-04-14",
    model_params={
        "temperature": 0,
        "response_format": {"type": "json_object"},
    },
)

embedder = OpenAIEmbeddings(model="text-embedding-3-large")

print("LLM and Embedder ready!")

## Step 3: Define Your Text

Here's a sample text about the history of Transformer models in AI. We'll extract knowledge from this.

In [3]:
TEXT = """
The Transformer architecture was introduced by Vaswani et al. in 2017 at Google Brain.
It revolutionized natural language processing by replacing recurrent neural networks with
self-attention mechanisms. The key innovation was the multi-head attention mechanism,
which allows the model to attend to different positions in the input sequence simultaneously.

BERT (Bidirectional Encoder Representations from Transformers) was developed by Jacob Devlin
and his colleagues at Google AI Language in 2018. BERT uses the encoder portion of the
Transformer architecture and introduced masked language modeling as a pre-training objective.
BERT achieved state-of-the-art results on eleven NLP benchmarks including GLUE, SQuAD, and
MultiNLI.

GPT (Generative Pre-trained Transformer) was created by OpenAI. The first version, GPT-1,
was released in 2018 by Alec Radford and colleagues. GPT uses the decoder portion of the
Transformer architecture. GPT-2 was released in 2019 with 1.5 billion parameters, and GPT-3
followed in 2020 with 175 billion parameters. GPT-4 was released in March 2023.

Transfer learning in NLP was pioneered by models like ELMo (developed at Allen AI by
Matthew Peters in 2018), ULMFiT (developed by Jeremy Howard and Sebastian Ruder), and
later by BERT and GPT. These models demonstrated that pre-training on large text corpora
and fine-tuning on specific tasks could dramatically improve performance.
"""

print(f"Text length: {len(TEXT)} characters")
print(f"Preview: {TEXT[:100]}...")

Text length: 1421 characters
Preview: 
The Transformer architecture was introduced by Vaswani et al. in 2017 at Google Brain.
It revolutio...


## Step 4: Define the Schema (Optional)

The schema tells the LLM what **types** of entities and relationships to look for. Providing a schema gives you more consistent, targeted results — but it's entirely optional.

**If you don't have a schema**, set `USE_MANUAL_SCHEMA = False` below. The LLM will read a sample of the text, propose its own schema, and then use it consistently across all chunks (`schema="EXTRACTED"`, the default).

**If you do have a schema**, define your node types, relationship types, and patterns below. Think of it like giving the LLM a template:
- **Node types**: What kinds of things exist? (People, Organizations, Models...)
- **Relationship types**: How are they connected? (DEVELOPED_BY, BASED_ON...)
- **Patterns**: Which connections are valid? (Person WORKS_AT Organization)

In [ ]:
# ── Toggle: set to False to let the LLM generate its own schema ──
USE_MANUAL_SCHEMA = True

# Define the types of entities we want to extract
NODE_TYPES = [
    # Simple string = just a label (automatically gets a "name" property)
    "Person",
    # Dict = label + description + properties for more control
    # NOTE: Dict-based node types MUST include at least one property.
    # When you use a plain string like "Person", it auto-gets a "name" property.
    # When using a dict, you need to include it explicitly.
    {
        "label": "Organization",
        "description": "A company, university, or research lab",
        "properties": [{"name": "name", "type": "STRING"}],
    },
    {
        "label": "AIModel",
        "description": "An AI/ML model or architecture",
        "properties": [
            {"name": "name", "type": "STRING"},
            {"name": "year_released", "type": "INTEGER"},
            {"name": "num_parameters", "type": "STRING"},
        ],
    },
    {
        "label": "Benchmark",
        "description": "An evaluation benchmark or dataset",
        "properties": [{"name": "name", "type": "STRING"}],
    },
]

# Define the types of relationships
RELATIONSHIP_TYPES = [
    "DEVELOPED_BY",
    "WORKS_AT",
    {"label": "BASED_ON", "description": "One model is built on top of another"},
    "EVALUATED_ON",
    "SUCCESSOR_OF",
]

# Define valid patterns (which node types can be connected by which relationships)
PATTERNS = [
    ("AIModel", "DEVELOPED_BY", "Person"),
    ("AIModel", "DEVELOPED_BY", "Organization"),
    ("Person", "WORKS_AT", "Organization"),
    ("AIModel", "BASED_ON", "AIModel"),
    ("AIModel", "EVALUATED_ON", "Benchmark"),
    ("AIModel", "SUCCESSOR_OF", "AIModel"),
]

if USE_MANUAL_SCHEMA:
    print(f"Using MANUAL schema: {len(NODE_TYPES)} node types, {len(RELATIONSHIP_TYPES)} relationship types, {len(PATTERNS)} patterns")
else:
    print("Using AUTO schema: the LLM will extract a schema from the text")

## Step 5: Build the Knowledge Graph!

Now the magic happens. `SimpleKGPipeline` will:
1. Split the text into chunks
2. *(If auto-schema)* Read a sample and propose a schema
3. Send each chunk to the LLM with the schema
4. Parse the LLM's response into nodes and relationships
5. Write everything to Neo4j
6. Resolve duplicate entities (merge nodes that refer to the same thing)

In [ ]:
from neo4j_graphrag.experimental.pipeline.kg_builder import SimpleKGPipeline

# Build the schema parameter based on the toggle
if USE_MANUAL_SCHEMA:
    schema = {
        "node_types": NODE_TYPES,
        "relationship_types": RELATIONSHIP_TYPES,
        "patterns": PATTERNS,
    }
else:
    # "EXTRACTED" = LLM reads a sample and proposes a schema automatically
    # This is actually the default, so schema=None would also work
    schema = "EXTRACTED"

kg_builder = SimpleKGPipeline(
    llm=llm,
    driver=driver,
    embedder=embedder,
    schema=schema,
    from_pdf=False,  # We're passing text, not a PDF
    neo4j_database=NEO4J_DATABASE,
)

print("Pipeline created. Running...")

# Run the pipeline (this is async, so we use await)
result = await kg_builder.run_async(text=TEXT)

print("\nKnowledge graph built successfully!")
print(f"Result: {result}")

## Step 6: Explore the Graph

Let's query Neo4j to see what was extracted.

In [6]:
# What node types were created?
records, _, _ = driver.execute_query(
    """
    MATCH (n)
    WHERE NOT n:Chunk AND NOT n:Document
    WITH labels(n) AS node_labels, count(n) AS count
    RETURN node_labels, count
    ORDER BY count DESC
    """,
    database_=NEO4J_DATABASE,
)

print("=== Node Types ===")
for record in records:
    print(f"  {record['node_labels']}: {record['count']} nodes")

=== Node Types ===
  ['__KGBuilder__', 'AIModel', '__Entity__']: 9 nodes
  ['Person', '__KGBuilder__', '__Entity__']: 6 nodes
  ['__KGBuilder__', '__Entity__', 'Organization']: 4 nodes
  ['__KGBuilder__', '__Entity__', 'Benchmark']: 3 nodes


In [7]:
# What relationships were created?
records, _, _ = driver.execute_query(
    """
    MATCH ()-[r]->()
    WHERE type(r) <> 'FROM_DOCUMENT' AND type(r) <> 'NEXT_CHUNK' AND type(r) <> 'FROM_CHUNK'
    RETURN type(r) AS rel_type, count(r) AS count
    ORDER BY count DESC
    """,
    database_=NEO4J_DATABASE,
)

print("=== Relationship Types ===")
for record in records:
    print(f"  {record['rel_type']}: {record['count']} relationships")

=== Relationship Types ===
  DEVELOPED_BY: 11 relationships
  EVALUATED_ON: 3 relationships
  SUCCESSOR_OF: 3 relationships
  BASED_ON: 2 relationships
  WORKS_AT: 1 relationships


In [8]:
# Let's see all extracted entities and their connections!
records, _, _ = driver.execute_query(
    """
    MATCH (a)-[r]->(b)
    WHERE NOT a:Chunk AND NOT a:Document AND NOT b:Chunk AND NOT b:Document
    RETURN labels(a)[0] AS from_type, a.name AS from_name,
           type(r) AS relationship,
           labels(b)[0] AS to_type, b.name AS to_name
    ORDER BY from_type, from_name
    """,
    database_=NEO4J_DATABASE,
)

print("=== Knowledge Graph Triples ===")
for record in records:
    print(
        f"  ({record['from_type']}: {record['from_name']}) "
        f"-[{record['relationship']}]-> "
        f"({record['to_type']}: {record['to_name']})"
    )

=== Knowledge Graph Triples ===
  (Person: Jacob Devlin) -[WORKS_AT]-> (__KGBuilder__: Google AI Language)
  (__KGBuilder__: BERT) -[EVALUATED_ON]-> (__KGBuilder__: SQuAD)
  (__KGBuilder__: BERT) -[EVALUATED_ON]-> (__KGBuilder__: GLUE)
  (__KGBuilder__: BERT) -[DEVELOPED_BY]-> (__KGBuilder__: Google AI Language)
  (__KGBuilder__: BERT) -[EVALUATED_ON]-> (__KGBuilder__: MultiNLI)
  (__KGBuilder__: BERT) -[DEVELOPED_BY]-> (Person: Jacob Devlin)
  (__KGBuilder__: BERT) -[BASED_ON]-> (__KGBuilder__: Transformer)
  (__KGBuilder__: ELMo) -[DEVELOPED_BY]-> (__KGBuilder__: Allen AI)
  (__KGBuilder__: ELMo) -[DEVELOPED_BY]-> (Person: Matthew Peters)
  (__KGBuilder__: GPT) -[BASED_ON]-> (__KGBuilder__: Transformer)
  (__KGBuilder__: GPT) -[DEVELOPED_BY]-> (__KGBuilder__: OpenAI)
  (__KGBuilder__: GPT-1) -[DEVELOPED_BY]-> (Person: Alec Radford)
  (__KGBuilder__: GPT-1) -[DEVELOPED_BY]-> (__KGBuilder__: OpenAI)
  (__KGBuilder__: GPT-2) -[SUCCESSOR_OF]-> (__KGBuilder__: GPT-1)
  (__KGBuilder__: GPT

## Step 7: Try a Cypher Query

Now that we have a knowledge graph, we can ask structured questions using Cypher (Neo4j's query language).

In [9]:
# Find all AI models and who developed them
records, _, _ = driver.execute_query(
    """
    MATCH (model:AIModel)-[:DEVELOPED_BY]->(developer)
    RETURN model.name AS model, labels(developer)[0] AS developer_type, developer.name AS developer
    ORDER BY model.name
    """,
    database_=NEO4J_DATABASE,
)

print("=== AI Models and Their Developers ===")
for record in records:
    print(f"  {record['model']} → developed by {record['developer']} ({record['developer_type']})")

=== AI Models and Their Developers ===
  BERT → developed by Google AI Language (__KGBuilder__)
  BERT → developed by Jacob Devlin (Person)
  ELMo → developed by Allen AI (__KGBuilder__)
  ELMo → developed by Matthew Peters (Person)
  GPT → developed by OpenAI (__KGBuilder__)
  GPT-1 → developed by Alec Radford (Person)
  GPT-1 → developed by OpenAI (__KGBuilder__)
  Transformer → developed by Google Brain (__KGBuilder__)
  Transformer → developed by Vaswani (Person)
  ULMFiT → developed by Jeremy Howard (Person)
  ULMFiT → developed by Sebastian Ruder (Person)


In [10]:
# Find chains of models (what's built on what?)
records, _, _ = driver.execute_query(
    """
    MATCH path = (a:AIModel)-[:BASED_ON|SUCCESSOR_OF*1..3]->(b:AIModel)
    RETURN a.name AS model, b.name AS based_on
    """,
    database_=NEO4J_DATABASE,
)

print("=== Model Lineage ===")
if records:
    for record in records:
        print(f"  {record['model']} is based on / successor of → {record['based_on']}")
else:
    print("  No model lineage relationships found (this depends on what the LLM extracted)")

=== Model Lineage ===
  BERT is based on / successor of → Transformer
  GPT is based on / successor of → Transformer
  GPT-2 is based on / successor of → GPT-1
  GPT-3 is based on / successor of → GPT-2
  GPT-3 is based on / successor of → GPT-1
  GPT-4 is based on / successor of → GPT-3
  GPT-4 is based on / successor of → GPT-2
  GPT-4 is based on / successor of → GPT-1


## Bonus: Try With Your Own Text

Replace the text below with anything you like and re-run!

In [ ]:
MY_TEXT = """
Replace this with your own text! For example, paste in a Wikipedia paragraph,
a news article, or any text you want to extract knowledge from.
"""

# You can reuse the same pipeline or create a new one with a different schema
# result = await kg_builder.run_async(text=MY_TEXT)

## Summary

In this notebook you learned how to:
- Define a schema with node types, relationship types, and patterns
- Use `SimpleKGPipeline` to extract knowledge from plain text
- Query the resulting knowledge graph with Cypher

**Key takeaway:** The schema is your most powerful tool for controlling what gets extracted. A well-defined schema produces a cleaner, more useful graph.

**Next up:** [Notebook 3 — Building a Knowledge Graph from PDFs](./03_kg_from_pdf.ipynb)

In [11]:
driver.close()